# Homework 2 · Data Profiling ข้อมูลลูกค้า (customers.csv)

ต่อยอดจากแนวทางของ Lab 2.1 (Data Profiling และทำความสะอาดข้อมูลด้วย AI) แต่เปลี่ยนชุดข้อมูลจากยอดขายเป็น **ข้อมูลลูกค้า** (`customers.csv`) เป้าหมายคือค้นหาความผิดปกติ (ถ้ามี) แล้วส่งออก `customers_clean.csv` ไปแสดงผลต่อในเว็บ Baanbrew Dashboard

**กำหนดส่ง:** ศุกร์ 2 ต.ค. 2569 ก่อนเที่ยงคืน — ส่งไฟล์ `.ipynb` นี้ (รันจริงใน Google Colab) และลิงก์เว็บ Baanbrew Dashboard ที่เพิ่มการแสดงผลข้อมูลลูกค้าแล้ว

**Prompt ต้นแบบของการบ้านนี้:**
> ช่วยทำ data profiling หรือค้นหาความผิดปกติของข้อมูล customers.csv แล้วสร้างเป็นไฟล์สำหรับทำงานบน google colab ดังตัวอย่าง Lab2_1_Data_Cleaning_NK.ipynb

**วิธีใช้กับ AI:** ใช้ Gemini ที่อยู่ใน Colab หรือคัดลอก prompt ไปใช้กับ Claude / ChatGPT แล้ววางโค้ดกลับมา

## ขั้นที่ 0 · เตรียมเครื่องมือ
รันเซลล์นี้ก่อน (ไม่ต้องแก้)

In [ ]:
# ===== เซลล์ตัวช่วย · รันก่อน ไม่ต้องแก้ =====
import os
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)


def load_csv(name):
    """อ่าน CSV ทุกคอลัมน์เป็นข้อความ ถ้าไม่มีไฟล์จะให้อัปโหลด (Google Colab)"""
    if not os.path.exists(name):
        try:
            from google.colab import files
            print(f"ไม่พบ {name} กรุณาอัปโหลดไฟล์")
            files.upload()
        except ImportError:
            raise FileNotFoundError(f"ไม่พบไฟล์ {name} ให้วางไฟล์ไว้โฟลเดอร์เดียวกับ notebook")
    return pd.read_csv(name, dtype=str, keep_default_na=False)

## ขั้นที่ 1 · ดูข้อมูลครั้งแรก (5 นาที)

อัปโหลด `customers.csv` และ `branches.csv` จากโฟลเดอร์นี้ (ถ้าใช้ Colab จะมีปุ่มให้อัปโหลด) — `branches.csv` ใช้ตรวจว่า `home_branch_id` แต่ละแถวมีสาขาอยู่จริงไหม

เราอ่าน **ทุกคอลัมน์เป็นข้อความ** โดยตั้งใจ เหมือนใน Lab 2.1 เพื่อไม่ให้ pandas เดาชนิดข้อมูลเองจนซ่อนปัญหาบางอย่างไป

In [ ]:
df = load_csv("customers.csv")
branches = load_csv("branches.csv")
print(f"{len(df):,} แถว × {df.shape[1]} คอลัมน์")
df.head(8)

## ขั้นที่ 2 · Profiling: หาปัญหาก่อน ยังไม่แก้ (10 นาที)

**Prompt**
```
ช่วยทำ data profiling ของ DataFrame df ด้วย pandas (ทุกคอลัมน์เป็น str ค่าว่างเป็น "")
- จำนวนค่าว่างและค่าไม่ซ้ำของแต่ละคอลัมน์
- จำนวนแถวที่ customer_id ซ้ำ
- ค่าที่พบทั้งหมดของ gender และ age_group พร้อมจำนวนแถว (มีค่าที่สะกดต่างกันไหม)
- จำนวน home_branch_id ที่ไม่ตรงกับสาขาจริงใน branches.csv (คอลัมน์ branch_id)
- จำนวน joined_date ที่แปลงเป็นวันที่ไม่ได้ หรือเกิดก่อนวันที่สาขานั้นเปิด (ดู opened_date ใน branches.csv)
- จำนวนเบอร์โทรที่ไม่ตรงรูปแบบ 0XX-xxx-XXXX
อย่าแก้ข้อมูล ให้รายงานอย่างเดียว
```

ทำไมต้อง "รายงานก่อน แก้ทีหลัง": ถ้าสั่งแก้ทันที AI อาจลบหรือเปลี่ยนข้อมูลเงียบ ๆ โดยเราไม่รู้ว่าเปลี่ยนอะไรไปบ้าง

In [ ]:
print("จำนวนค่าว่างต่อคอลัมน์")
print(df.eq("").sum())
print()

print("จำนวนค่าไม่ซ้ำต่อคอลัมน์")
print(df.nunique())
print()

n_dup_id = df["customer_id"].duplicated().sum()
print(f"customer_id ที่ซ้ำ: {n_dup_id}")
print()

print("gender:")
print(df["gender"].value_counts())
print()
print("age_group:")
print(df["age_group"].value_counts())
print()

valid_branches = set(branches["branch_id"])
bad_branch = ~df["home_branch_id"].isin(valid_branches)
print(f"home_branch_id ที่ไม่ตรงสาขาจริง: {bad_branch.sum()}")

joined = pd.to_datetime(df["joined_date"], errors="coerce")
bad_date = joined.isna()
print(f"joined_date แปลงเป็นวันที่ไม่ได้: {bad_date.sum()}")

opened = dict(zip(branches["branch_id"], pd.to_datetime(branches["opened_date"])))
before_open = 0
for i, r in df.iterrows():
    b = r["home_branch_id"]
    if b in opened and pd.notna(joined[i]) and joined[i] < opened[b]:
        before_open += 1
print(f"joined_date ก่อนวันที่สาขานั้นเปิด: {before_open}")

bad_phone = ~df["phone"].str.match(r"^0\d\d-xxx-\d{4}$")
print(f"เบอร์โทรรูปแบบผิดปกติ: {bad_phone.sum()}")

## ขั้นที่ 3 · สรุปผล profiling และตัดสินใจ (5 นาที)

หลังรันเซลล์ด้านบนแล้ว สรุปผลและตัดสินใจว่าจะจัดการแต่ละจุดอย่างไร (ดับเบิลคลิกเซลล์นี้เพื่อกรอกตาราง — ใส่ตัวเลขจริงที่ได้จากขั้นที่ 2)

| ประเด็นที่ตรวจ | พบปัญหากี่แถว | ควรจัดการอย่างไร | เหตุผล |
|---|---|---|---|
| ค่าว่าง / customer_id ซ้ำ | | | |
| gender มีค่า "ไม่ระบุ" | | เก็บไว้เป็นหมวดหนึ่ง / ตัดทิ้ง | ลูกค้าบางคนไม่กรอกข้อมูลจริง ไม่ใช่ข้อผิดพลาด |
| age_group สะกดไม่ตรงมาตรฐาน | | | |
| home_branch_id ไม่ตรงสาขาจริง | | | |
| joined_date แปลงไม่ได้ / ก่อนสาขาเปิด | | | |
| เบอร์โทรผิดรูปแบบ | | | |

## ขั้นที่ 4 · ทำความสะอาดด้วย AI (ถ้าจำเป็น)

**Prompt ตั้งต้น** (แก้ส่วนการตัดสินใจให้ตรงกับตารางของคุณในขั้นที่ 3)

```
เขียนโค้ด pandas ทำความสะอาด df (อ่านมาเป็น str ทุกคอลัมน์ ค่าว่างเป็น "") ตามผล profiling ด้านบน
- ทำงานบน clean = df.copy() ห้ามแก้ df
- แปลง joined_date เป็นรูปแบบ YYYY-MM-DD ที่แปลงเป็นวันที่ได้จริงทุกแถว
- เรียง age_group เป็น categorical ตามลำดับ: ต่ำกว่า 18, 18-24, 25-34, 35-44, 45-54, 55+
- [เพิ่มการตัดสินใจอื่นตามตารางขั้นที่ 3 เช่น จัดการ home_branch_id ที่ไม่ตรงสาขา ถ้าพบ]
- เก็บ log เป็นรายการ (ขั้นตอน, จำนวนแถวที่กระทบ)
- แสดงจำนวนแถวก่อน/หลัง
```

In [ ]:
clean = df.copy()

# แปลง joined_date ให้เป็นรูปแบบมาตรฐาน
clean["joined_date"] = pd.to_datetime(clean["joined_date"], errors="coerce").dt.strftime("%Y-%m-%d")

# เรียง age_group ตามลำดับอายุ (ใช้กับกราฟ/ตารางใน Dashboard ภายหลัง)
age_order = ["ต่ำกว่า 18", "18-24", "25-34", "35-44", "45-54", "55+"]
clean["age_group"] = pd.Categorical(clean["age_group"], categories=age_order, ordered=True)

# เพิ่มโค้ดจัดการปัญหาอื่นที่พบจริงในขั้นที่ 2 ตรงนี้ (ถ้ามี)

print(f"ก่อนทำความสะอาด: {len(df):,} แถว · หลัง: {len(clean):,} แถว")
clean.head()

## ขั้นที่ 5 · Verify และ Export

ตรวจก่อน export ว่า:
- `joined_date` แปลงเป็นวันที่ได้ครบทุกแถว (ไม่มี NaT)
- `age_group` มีค่าอยู่ใน 6 หมวดมาตรฐานเท่านั้น
- จำนวนแถวหลังทำความสะอาดสมเหตุสมผล (ไม่หายไปเยอะผิดปกติถ้าไม่ได้ตั้งใจตัดทิ้ง)

In [ ]:
assert clean["joined_date"].notna().all(), "มี joined_date ที่แปลงเป็นวันที่ไม่ได้"
assert clean["age_group"].isin(age_order).all(), "มี age_group ที่ไม่อยู่ในหมวดมาตรฐาน"
print(f"✅ พร้อม export: {len(clean):,} แถว")

In [ ]:
clean.to_csv("customers_clean.csv", index=False, encoding="utf-8-sig")
print("บันทึก customers_clean.csv แล้ว")

try:
    from google.colab import files
    files.download("customers_clean.csv")
except ImportError:
    pass

### นำไปใช้ต่อใน Baanbrew Dashboard
1. คัดลอก `customers_clean.csv` ไปวางใน `public/` ของโปรเจกต์ baanbrew-dashboard แล้วเปลี่ยนชื่อเป็น `customers.csv`
2. เพิ่มส่วนแสดงผลข้อมูลลูกค้าในเว็บ (เช่น สัดส่วนตาม gender / age_group, จำนวนลูกค้าใหม่ต่อเดือนตาม `joined_date`, จำนวนลูกค้าต่อสาขาบ้าน)
3. Deploy ใหม่แล้วส่งลิงก์เว็บที่อัปเดตแล้ว